In [1]:
import requests
import pandas as pd
import re
import shutil

from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path

In [2]:
# المسارات
BASE_DIR = Path("..")
RAW_DATA_DIR = BASE_DIR / "raw_data"
RAW_IMAGES_DIR = BASE_DIR / "raw_images"
CLEANED_DATA_DIR = BASE_DIR / "cleaned_data"
CLEANED_IMAGES_DIR = BASE_DIR / "cleaned_images"

# إنشاء المجلدات
RAW_DATA_DIR.mkdir(exist_ok=True)
RAW_IMAGES_DIR.mkdir(exist_ok=True)
CLEANED_DATA_DIR.mkdir(exist_ok=True)
CLEANED_IMAGES_DIR.mkdir(exist_ok=True)

# إنشاء مجلدات التقييمات للصور المنظفة
for rating in range(1, 6):
    (CLEANED_IMAGES_DIR / str(rating)).mkdir(parents=True, exist_ok=True)

print("Folders are ready.")

Folders are ready.


In [3]:
BASE_URL = "http://books.toscrape.com/"
START_URL = "http://books.toscrape.com/catalogue/category/books_1/page-1.html"

MAX_PAGES = 3

In [4]:
def safe_filename(name):
    """
    تنظيف اسم الملف من الرموز غير المناسبة لنظام الملفات
    """
    return re.sub(r'[\\/*?:"<>|]', "_", name)


def clean_price(price_text):
    """
    تحويل السعر من نص مثل £51.77 إلى float
    """
    cleaned = re.sub(r"[^\d.]", "", price_text)
    return float(cleaned)


def convert_rating_to_int(rating_text):
    """
    تحويل التقييم من One/Two/... إلى رقم من 1 إلى 5
    """
    rating_map = {
        "One": 1,
        "Two": 2,
        "Three": 3,
        "Four": 4,
        "Five": 5
    }
    return rating_map.get(rating_text, None)

In [5]:
raw_books = []

current_page = 1
url = START_URL

while url and current_page <= MAX_PAGES:
    response = requests.get(url)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    books = soup.find_all("article", class_="product_pod")

    for book in books:
        name = book.h3.a["title"].strip()
        price = book.find("p", class_="price_color").text.strip()
        rating = book.find("p", class_="star-rating")["class"][1]

        img_src = book.find("img")["src"]
        img_url = urljoin(url, img_src)

        image_filename = safe_filename(name) + ".jpg"
        image_path = RAW_IMAGES_DIR / image_filename

        # تحميل الصورة
        image_response = requests.get(img_url)
        image_response.raise_for_status()

        with open(image_path, "wb") as f:
            f.write(image_response.content)

        raw_books.append({
            "name": name,
            "price": price,
            "rating": rating,
            "image_path": str(image_path)
        })

    print(f"Page {current_page} scraped successfully.")

    current_page += 1
    next_button = soup.find("li", class_="next")

    if next_button and current_page <= MAX_PAGES:
        url = f"http://books.toscrape.com/catalogue/category/books_1/page-{current_page}.html"
    else:
        url = None

print("Raw scraping completed.")

Page 1 scraped successfully.
Page 2 scraped successfully.
Page 3 scraped successfully.
Raw scraping completed.


In [6]:
raw_df = pd.DataFrame(raw_books)

raw_csv_path = RAW_DATA_DIR / "raw_data.csv"
raw_df.to_csv(raw_csv_path, index=False, encoding="utf-8-sig")

print(f"Raw data saved to: {raw_csv_path}")
raw_df.head()

Raw data saved to: ..\raw_data\raw_data.csv


,name,price,rating,image_path
0,A Light in the Attic,Â£51.77,Three,..\raw_images\A Light in the Attic.jpg
1,Tipping the Velvet,Â£53.74,One,..\raw_images\Tipping the Velvet.jpg
2,Soumission,Â£50.10,One,..\raw_images\Soumission.jpg
3,Sharp Objects,Â£47.82,Four,..\raw_images\Sharp Objects.jpg
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,..\raw_images\Sapiens_ A Brief History of Huma...


In [7]:
cleaned_df = raw_df.copy()

cleaned_df["name"] = cleaned_df["name"].astype(str)
cleaned_df["price"] = cleaned_df["price"].apply(clean_price)
cleaned_df["rating"] = cleaned_df["rating"].apply(convert_rating_to_int).astype(int)

cleaned_df.head()

,name,price,rating,image_path
0,A Light in the Attic,51.77,3,..\raw_images\A Light in the Attic.jpg
1,Tipping the Velvet,53.74,1,..\raw_images\Tipping the Velvet.jpg
2,Soumission,50.10,1,..\raw_images\Soumission.jpg
3,Sharp Objects,47.82,4,..\raw_images\Sharp Objects.jpg
4,Sapiens: A Brief History of Humankind,54.23,5,..\raw_images\Sapiens_ A Brief History of Huma...


In [8]:
for rating_value in sorted(cleaned_df["rating"].unique()):
    rating_df = cleaned_df[cleaned_df["rating"] == rating_value].copy()

    output_csv = CLEANED_DATA_DIR / f"{rating_value}.csv"
    rating_df.to_csv(output_csv, index=False, encoding="utf-8-sig")

    print(f"Saved: {output_csv}")

Saved: ..\cleaned_data\1.csv
Saved: ..\cleaned_data\2.csv
Saved: ..\cleaned_data\3.csv
Saved: ..\cleaned_data\4.csv
Saved: ..\cleaned_data\5.csv


In [9]:
for _, row in cleaned_df.iterrows():
    rating = row["rating"]
    old_image_path = Path(row["image_path"])

    new_image_path = CLEANED_IMAGES_DIR / str(rating) / old_image_path.name

    shutil.copy(old_image_path, new_image_path)

print("Images organized by rating successfully.")

Images organized by rating successfully.


In [10]:
for rating_value in sorted(cleaned_df["rating"].unique()):
    rating_df = cleaned_df[cleaned_df["rating"] == rating_value].copy()

    updated_paths = []
    for _, row in rating_df.iterrows():
        old_image_path = Path(row["image_path"])
        new_image_path = CLEANED_IMAGES_DIR / str(rating_value) / old_image_path.name
        updated_paths.append(str(new_image_path))

    rating_df["image_path"] = updated_paths

    output_csv = CLEANED_DATA_DIR / f"{rating_value}.csv"
    rating_df.to_csv(output_csv, index=False, encoding="utf-8-sig")

    print(f"Updated and saved: {output_csv}")

Updated and saved: ..\cleaned_data\1.csv
Updated and saved: ..\cleaned_data\2.csv
Updated and saved: ..\cleaned_data\3.csv
Updated and saved: ..\cleaned_data\4.csv
Updated and saved: ..\cleaned_data\5.csv
